# Interactive Time-Series Explorer

Pick a **dataset** (soil or met), a **station**, then a **variable** to see its
hourly time series. The dropdowns cascade: changing the dataset refills the
station list, and changing the station refills the variable list.

Wherever the **selected variable has no measurement (`NaN`)**, that stretch of
time is shaded as a **red block** on the plot. Because the data is pre-washed
(`prewash_the_data=True`), this captures both filled-in missing timestamps
*and* replaces impossible values with NA

### Requirements
Needs `plotly` and `ipywidgets`.

In [3]:
import os, sys
from pathlib import Path

# Directory containing this notebook.
NOTEBOOK_DIR = Path.cwd()

# Add the scripts folder to the import path, then import data_ingest.
sys.path.insert(0, str(NOTEBOOK_DIR / "scripts"))
from get_data_dict import data_ingest

# Raw TxSON data folder: the TXSON_DATA env var, or a default sibling path.
DATA_FOLDER = str(Path(os.environ.get(
    "TXSON_DATA",
    NOTEBOOK_DIR.parent / "datasets" / "TxSON_data_2026-02-24",
)))

# prewash_the_data fills missing hourly timestamps with NaN so gaps show as red blocks.
ingest = data_ingest(
    input_data_folder=DATA_FOLDER,
    prewash_the_data=True,
    clean_the_data=False,
)
met_dict, soil_dict = ingest.get_data_dict()

print(f"Met  stations: {len(met_dict):>2}  ->  {list(met_dict)}")
print(f"Soil stations: {len(soil_dict):>2}  ->  {list(soil_dict)}")

Met  stations:  6  ->  ['CB01_met', 'CB04_met', 'CB06_met', 'FD02_met', 'FD03_met', 'WC05_met']
Soil stations: 33  ->  ['CB01', 'CB04', 'CB06', 'CB07', 'CB09', 'CB10', 'CB15', 'CB19', 'CB20', 'CB25', 'CB26', 'CB27', 'CB31', 'CB32', 'CB33', 'FD02', 'FD03', 'FD08', 'FD11', 'FD12', 'FD13', 'FD14', 'FD16', 'FD17', 'FD18', 'FD21', 'FD22', 'FD23', 'FD24', 'FD28', 'FD29', 'FD30', 'WC05']


# AI CODE

In [6]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML

# Two source dictionaries keyed by station name. "Flag" is not a measurement.
DATASETS = {"Soil": soil_dict, "Met": met_dict}
IGNORE_COLS = {"Flag"}
NA_COLOR = "crimson"
LINE_COLOR = "#1f77b4"
# Where each render gets plotly.js: "cdn" (small, needs internet once) or
# True to inline it (fully offline, but bigger output).
PLOTLYJS = "cdn"

last_fig = None  # most recently drawn figure (handy for exporting the current view)


def na_blocks(mask, index):
    """Group a boolean NA mask into contiguous (start, end) timestamp spans."""
    m = np.asarray(mask, dtype=bool)
    if not m.any():
        return []
    edges = np.diff(np.concatenate(([0], m.astype(np.int8), [0])))
    starts = np.where(edges == 1)[0]
    ends = np.where(edges == -1)[0] - 1  # inclusive end of each run
    return [(index[s], index[e]) for s, e in zip(starts, ends)]


def classify_gaps(blocks, step):
    """Bucket NA gaps by duration: short <24h, medium 1-7d, long 7-30d, very long >30d."""
    day = pd.Timedelta(days=1)
    counts = {"short": 0, "medium": 0, "long": 0, "very_long": 0}
    for start, end in blocks:
        d = (end - start) + step  # gap span = missing samples x sampling step
        if d < day:
            counts["short"] += 1
        elif d < 7 * day:
            counts["medium"] += 1
        elif d <= 30 * day:
            counts["long"] += 1
        else:
            counts["very_long"] += 1
    return counts


def make_figure(dataset, station, variable, df):
    """Time-series figure for one variable, with NA stretches shaded red."""
    series = pd.to_numeric(df[variable], errors="coerce")
    mask = series.isna()
    blocks = na_blocks(mask.to_numpy(), df.index)

    # Representative sampling step (hourly data); used for gap length + padding.
    if len(df.index) > 1:
        step = pd.Timedelta(np.median(np.diff(df.index.values)))
    else:
        step = pd.Timedelta(hours=1)
    pad = step / 2  # widen each block so single-sample gaps stay visible

    shapes = [
        dict(type="rect", xref="x", yref="paper",
             x0=start - pad, x1=end + pad, y0=0, y1=1,
             fillcolor=NA_COLOR, opacity=0.30, line_width=0, layer="below")
        for start, end in blocks
    ]

    fig = go.Figure()
    fig.add_scatter(x=df.index, y=series.to_numpy(), mode="lines",
                    line=dict(color=LINE_COLOR, width=1),
                    name=variable, connectgaps=False)
    # Invisible trace so the red NA shading gets a legend entry.
    fig.add_scatter(x=[None], y=[None], mode="markers",
                    marker=dict(symbol="square", size=11,
                                color=NA_COLOR, opacity=0.30),
                    name="NA (no data)")

    g = classify_gaps(blocks, step)
    n_na, n = int(mask.sum()), len(series)
    pct = (100 * n_na / n) if n else 0.0
    fig.update_layout(
        title=dict(text=(
            f"{dataset} · {station} · {variable}   —   "
            f"{n_na:,} NA / {n:,} ({pct:.1f}%) in {len(blocks):,} gap(s)"
            f"<br><span style='font-size:12px;color:#666'>gap length →  "
            f"&lt;24h: {g['short']}   ·   1–7d: {g['medium']}   ·   "
            f"7–30d: {g['long']}   ·   &gt;30d: {g['very_long']}</span>"),
            x=0.5, xanchor="center"),
        shapes=shapes, template="plotly_white", height=480,
        margin=dict(l=60, r=20, t=92, b=40), hovermode="x unified",
        xaxis_title="Date", yaxis_title=variable,
        legend=dict(orientation="h", yanchor="bottom", y=1.02,
                    xanchor="right", x=1),
    )
    return fig


# ---- controls -------------------------------------------------------------
dataset_dd  = widgets.Dropdown(options=list(DATASETS), description="Dataset:")
station_dd  = widgets.Dropdown(description="Station:")
variable_dd = widgets.Dropdown(description="Variable:")
plot_out    = widgets.Output()
_updating   = {"on": False}  # suppress handlers during programmatic option changes


def _redraw(*_):
    global last_fig
    ds, station, variable = dataset_dd.value, station_dd.value, variable_dd.value
    df = DATASETS.get(ds, {}).get(station)
    if df is None or variable is None or variable not in df.columns:
        return
    last_fig = make_figure(ds, station, variable, df)
    html = last_fig.to_html(full_html=False, include_plotlyjs=PLOTLYJS)
    with plot_out:
        plot_out.clear_output(wait=True)
        display(HTML(html))


def _sync_variable_options():
    df = DATASETS.get(dataset_dd.value, {}).get(station_dd.value)
    cols = [c for c in df.columns if c not in IGNORE_COLS] if df is not None else []
    _updating["on"] = True
    variable_dd.options = cols
    variable_dd.value = cols[0] if cols else None
    _updating["on"] = False


def _sync_station_options():
    keys = list(DATASETS.get(dataset_dd.value, {}))
    _updating["on"] = True
    station_dd.options = keys
    station_dd.value = keys[0] if keys else None
    _updating["on"] = False


def _on_dataset_change(*_):
    if _updating["on"]:
        return
    _sync_station_options()
    _sync_variable_options()
    _redraw()


def _on_station_change(*_):
    if _updating["on"]:
        return
    _sync_variable_options()
    _redraw()


def _on_variable_change(*_):
    if _updating["on"]:
        return
    _redraw()


dataset_dd.observe(_on_dataset_change, names="value")
station_dd.observe(_on_station_change, names="value")
variable_dd.observe(_on_variable_change, names="value")

# Populate the cascade and draw the first view.
_sync_station_options()
_sync_variable_options()
_redraw()

display(widgets.VBox([
    widgets.HBox([dataset_dd, station_dd, variable_dd]),
    plot_out,
]))